In [1]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pickle
# ==========================================================
# 1. 设置清晰、大气的演示风格
# ==========================================================
# 基础风格设置，优化字体、刻度等
sns.set_theme(style="ticks", context="talk")

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],
    'axes.labelsize': 24,
    'axes.linewidth': 2,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
    'xtick.major.width': 2,
    'ytick.major.width': 2,
    'lines.linewidth': 3,
    'lines.markersize': 14,  # 稍微增大标记以便看得更清楚
    'legend.fontsize': 18,
    'legend.frameon': True, # 按照您的图例，重新加上边框
    'legend.edgecolor': 'gray', # 边框颜色设为灰色，不那么刺眼
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})
'''
# ==========================================================
# 2. 定义新的样式映射 (严格匹配您提供的图片)
# ==========================================================
# 手动定义颜色以精确匹配图例
colors = {
    'blue': '#1f77b4',
    'orange': '#ff7f0e',
    'green': '#2ca02c', # 鲜明的绿色
    'red': '#d62728',
    'black': 'black'
}

# 创建新的样式字典，严格匹配您的图例
# 标签名称保持与您的数据一致 ('VQE_noiseless' 等)
style_map = {
    'FCI': {
        'color': colors['black'], 'marker': 'o', 'linestyle': '-', 'label': 'FCI', 'lw': 3.5
    },
    'VQE_noiseless': {
        'color': colors['orange'], 'marker': '>', 'linestyle': '--', 'label': 'VQE-UCCSD'
    },
    'VQE_noisy': {
        'color': colors['green'], 'marker': 'p', 'linestyle': '--', 'label': 'VQE-UCCSD (Noisy)'
    },
    'VRDM-DQG': {
        'color': colors['blue'], 'marker': 's', 'linestyle': '--', 'label': 'VRDM'
    },
    'E_thm_CDR': {
        # 将此项映射到图例中的 VQE+SDP 风格 (红橙色)，并使用独特的标记
        'color': colors['red'], 'marker': 'D', 'linestyle': ':', 'label': 'E_thm_CDR'
    },
}

# ==========================================================
# 2. 定义新的样式映射 (严格匹配您提供的图片)
# ==========================================================
# 手动定义颜色以精确匹配图例
# colors = {
#     'blue': '#1f77b4',
#     'orange': '#ff7f0e',
#     'green': '#2ca02c', # 鲜明的绿色
#     'red': '#d62728',
#     'black': 'black'
# }
'''
# 3. 使用色觉障碍者友好的色板
# colors = sns.color_palette('colorblind', 5)
# 方案一: Seaborn "muted"
# colors = sns.color_palette('muted', 5) 

# 方案二: Paul Tol's "bright" (直接使用十六进制代码)
# colors = ['#4477aa', '#ee6677', '#228833', '#ccbb44', '#66ccee']

# 方案三: Viridis (采样)
# colors = sns.color_palette('viridis', 5)

# color_map = {
#     'VRDM': colors[0], 'VQE-UCCSD': colors[1], 'Noisy': colors[2],
#    'FCI': 'black', 'VQE+SDP': colors[3]
# }
colors = [
    '#204BA1',  # 对应您图中的 'Flat'
    '#609FD8',  # 对应您图中的 'Evo'
    '#C8D890',  # 对应您图中的 'SzKS'
    '#AFC63B',  # 对应您图中的 'SzKL'
    '#6FB057'   # 对应您图中的 'SzF'
]
# 创建新的样式字典，严格匹配您的图例
# 标签名称保持与您的数据一致 ('VQE_noiseless' 等)
style_map = {
    'VQE_noisy': {
        'color': colors[0], 'marker': 'p', 'linestyle': '--', 'label': 'VQE-UCCSD (Noisy)'
    },
    'VQE_noiseless': {
        'color': colors[1], 'marker': '>', 'linestyle': '--', 'label': 'VQE-UCCSD'
    },
    'FCI': {
        'color': colors[2], 'marker': 'o', 'linestyle': '-', 'label': 'FCI', 'lw': 3.5
    },
    'VRDM-DQG': {
        'color': colors[3], 'marker': 's', 'linestyle': '--', 'label': 'v2RDM-DQG'
    },
    'E_thm_CDR': {
        # 将此项映射到图例中的 VQE+SDP 风格 (红橙色)，并使用独特的标记
        'color': colors[4], 'marker': 'D', 'linestyle': ':', 'label': 'VQE+vRDM'
    },
}

# ==========================================================
# 2. 数据加载与处理函数 (与之前修正版相同)
# ==========================================================
def load_pickle_files(filenames):
    """辅助函数：加载一系列 pickle 文件。"""
    all_data = {}
    print(f"--- 正在加载 {len(filenames)} 个文件... ---")
    for filename in filenames:
        try:
            with open(filename, 'rb') as f:
                all_data[filename] = pickle.load(f)
        except FileNotFoundError:
            print(f"⚠️ 警告: 文件未找到，已跳过 -> {filename}")
    return all_data
def process_pkl_data_from_two_sources_VQE(x_axis_d, molecule='H2', basis='sto-3g'):
    """
    核心处理函数：从两个独立的数据源加载并处理所有 .pkl 文件。
    (已修正逻辑)
    """
    d_list = x_axis_d
    
    d_values, y_vqe_noiseless_raw, y_vqe_noisy_raw = [], [], []
    
    for d_val in d_list:
        # 3. 使用与加载时完全相同的文件名作为键，来获取数据
        noiseless_filename = f"{molecule}_d{d_val}_result_noiseless.pkl"
        noisy_filename = f"{molecule}_d{d_val}_result_noisy.pkl"
        
        print(noiseless_filename) # 用于调试，检查文件名是否正确
        print(noisy_filename)   # 用于调试
        
        with open(noiseless_filename, 'rb') as f:
            noiseless_data = pickle.load(f)
        with open(noisy_filename, 'rb') as f:
            noisy_data = pickle.load(f)


        # print(noiseless_data) # 用于调试，检查是否加载成功
        # print(noisy_data)   # 用于调试
        
        if noiseless_data is None or noisy_data is None:
            print(f"⚠️ 警告: d={d_val} 的数据不完整或文件不存在，已跳过。")
            continue

        d_values.append(d_val)
        y_vqe_noiseless_raw.append(noiseless_data.optimal_value)
        y_vqe_noisy_raw.append(noisy_data.optimal_value)

    return y_vqe_noiseless_raw, y_vqe_noisy_raw

def process_pkl_data_from_two_sources(x_axis_d, molecule='H2', basis='sto-3g'):
    """核心处理函数：从两个独立的数据源加载并处理所有 .pkl 文件。"""
    d_list = x_axis_d
    fci_file_list = [f"{molecule}_{d}_{basis}.pkl" for d in d_list]
    loaded_fci_data = load_pickle_files(fci_file_list)
    cdr_file_list = [f"data_result_{molecule}_{d}_{basis}.pkl" for d in d_list]
    loaded_cdr_data = load_pickle_files(cdr_file_list)
    d_values, y_fci, y_vrdm_dqg, y_ethm_cdr_min = [], [], [], []
    print("\n--- 正在整合两个数据源的数据... ---")
    for d_val in d_list:
        fci_filename = f"{molecule}_{d_val}_{basis}.pkl"
        cdr_filename = f"data_result_{molecule}_{d_val}_{basis}.pkl"
        fci_data = loaded_fci_data.get(fci_filename)
        cdr_data = loaded_cdr_data.get(cdr_filename)
        if fci_data is None or cdr_data is None:
            print(f"⚠️ 警告: d={d_val} 的数据不完整，已跳过。")
            continue
        d_values.append(d_val)
        y_fci.append(fci_data.get('FCI_val'))
        y_vrdm_dqg.append(cdr_data.get('E_thm_max'))
        if 'cdr_iterations' in cdr_data:
            iterations_list = cdr_data['cdr_iterations']
            energy_list = [item.get('E_thm_CDR') for item in iterations_list if item.get('E_thm_CDR') is not None]
            y_ethm_cdr_min.append(min(energy_list) if energy_list else None)
        else:
            y_ethm_cdr_min.append(None)
    return d_values, y_fci, y_vrdm_dqg, y_ethm_cdr_min


# ==========================================================
# 3. 主程序：整合、计算并绘制双面板图
# ==========================================================
if __name__ == "__main__":
    
    # --- 数据准备 ---
    x_axis_d = [0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.6, 1.9, 2.2, 2.5, 2.8, 3.1, 3.4]
    # y_vqe_noiseless_raw = [-1.02678321, -1.65311066, -1.96018772, -2.10698961, -2.16754608, -2.180281934, -2.16630798, -2.137805086, -2.10229385, -2.064677026, -1.965615734, -1.9053901, -1.8793227, -1.86789221, -1.86665314, -1.86628435, -1.8660219]
    # y_vqe_noisy_raw = [5.3276245, 3.09924288, 1.71171468, 0.85060199, 0.26763367, -0.092784486, -0.37284138, -0.536942283, -0.69694246, -0.81158268, -0.99202507, -1.06051044, -1.07821144, -1.09258275, -1.10101234, -1.09716453, -1.09578524]
    
    y_vqe_noiseless_raw, y_vqe_noisy_raw =process_pkl_data_from_two_sources_VQE(x_axis_d)
    print(y_vqe_noiseless_raw)
    print(y_vqe_noisy_raw)
    d_from_pkl, y_fci_raw, y_vrdm_dqg_raw, y_ethm_cdr_raw = process_pkl_data_from_two_sources(x_axis_d)
    
    # 数据对齐，确保所有数组长度一致
    aligned_data = {}
    valid_indices = [i for i, d in enumerate(x_axis_d) if d in d_from_pkl]
    aligned_data['d'] = np.array([x_axis_d[i] for i in valid_indices])
    aligned_data['VQE_noisy'] = np.array([y_vqe_noisy_raw[i] for i in valid_indices])
    aligned_data['VQE_noiseless'] = np.array([y_vqe_noiseless_raw[i] for i in valid_indices])
    aligned_data['FCI'] = np.array([y_fci_raw[d_from_pkl.index(x_axis_d[i])] for i in valid_indices])

    aligned_data['VRDM-DQG'] = np.array([y_vrdm_dqg_raw[d_from_pkl.index(x_axis_d[i])] for i in valid_indices])
    aligned_data['E_thm_CDR'] = np.array([y_ethm_cdr_raw[d_from_pkl.index(x_axis_d[i])] for i in valid_indices])

    # --- 创建双面板子图 ---
    # 2行1列，共享x轴，设置总画布大小
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 12), sharex=True, gridspec_kw={'height_ratios': [2, 2]})

    # --- 绘制顶部面板 (能量图) ---
    # ax1.set_title("H2 Dissociation and Error Mitigation", fontsize=18, pad=15)
    for name, data in aligned_data.items():
        if name != 'd':
            ax1.plot(aligned_data['d'], data, **style_map[name])
    ax1.set_ylabel("Energy (Hartree)")
    # ax1.grid(True, which='major', linestyle='--', linewidth=0.5)
    ax1.legend()

    # --- 绘制底部面板 (对数误差图) ---
    for name, data in aligned_data.items():
        if name not in ['d', 'FCI']:
            error = np.abs(data - aligned_data['FCI'])
            ax2.plot(aligned_data['d'], error, **style_map[name])
    
    ax2.set_yscale('log')
    ax2.axhline(y=1.6e-3, color='grey', linestyle='-.', linewidth=1.5, label='Chemical Accuracy')
    ax2.set_xlabel("H2 Dissociation distance (Å)") # X轴标签只在最下方显示
    ax2.set_ylabel("Absolute Error (Hartree)")
    # ax2.grid(True, which='major', linestyle='--', linewidth=0.5)
    # ax2.legend() # 误差图也显示图例，以明确曲线

    # --- 最终调整与输出 ---
    plt.tight_layout(pad=2.0) # 调整子图间距
    
    # 保存为高质量矢量图
    output_filename = "H2_publication_plot.svg"
    plt.savefig(output_filename, format='svg', bbox_inches='tight')
    print(f"\n✔️ 成功保存高质量矢量图到: {output_filename}")
    
    # 保存为高质量矢量图
    output_filename = "H2_publication_plot.pdf"
    plt.savefig(output_filename, format='pdf', bbox_inches='tight')
    print(f"\n✔️ 成功保存高质量矢量图到: {output_filename}")
    # 显示图形
    plt.show()

H2_d0.4_result_noiseless.pkl
H2_d0.4_result_noisy.pkl


AttributeError: Can't get attribute '_default_atomic_evolution' on <module 'qiskit.synthesis.evolution.product_formula' from '/opt/anaconda3/envs/GN002004/lib/python3.12/site-packages/qiskit/synthesis/evolution/product_formula.py'>